In [21]:
import os
os.environ.pop('HTTP_PROXY', None)
os.environ.pop('HTTPS_PROXY', None)
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)
from pathlib import Path
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [22]:
# 获取脚本所在目录
try:
    SCRIPT_DIR = Path(__file__).parent
except NameError:
    SCRIPT_DIR = Path.cwd()
DATA_DIR = SCRIPT_DIR / "data"
CHROMA_DIR = SCRIPT_DIR / "chroma_db"
# 确保 data目录存在
DATA_DIR.mkdir(exist_ok=True)
CHROMA_DIR.mkdir(exist_ok=True)
# 加载环境变量
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    raise ValueError("请设置环境变量 GROQ_API_KEY")
# 初始化模型
model = init_chat_model(model="groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

In [23]:
def example():
    # 准备文档
    test_content = """
    LangChain 框架核心组件
    1.Models - 模型接口
    支持 OpenAI, Anthropic, Groq 等多种模型。
    版本要求：langchain>=1.0.0

    2.Prompts - 提示词模板
    使用 PromptTemplate 和 ChatPromptTemplate。

    3.Agents - 智能代理
    使用 create_agent 函数创建代理。
    支持工具调用和 ReAct 模式。

    RAG 进阶技术
    1.混合检索 (Hybrid Search)
    结合向量搜索和 BM25 关键词搜索。

    2.BM25 算法
    Best Match 25，基于词频的检索算法。
    是 TF-IDF 的改进版本。

    3.EnsembleRetriever
    使用 RRF (Reciprocal Rank Fusion) 算法。
    组合多个检索器的结果。
    """

    # 加载文档
    test_file = DATA_DIR / "test_docs.txt"
    with open(test_file, "w", encoding="utf-8") as f:
        f.write(test_content)
    loader = TextLoader(test_file, encoding="utf-8")
    documents = loader.load()
    # 文档分块
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=150,
        chunk_overlap=30,
        separators=["\n\n", "\n", " ", ""],
    )
    chunks = splitter.split_documents(documents)
    print(f"\n[OK] 文档加载分块完成")
    print(f"原文档：{len(documents)} 个")
    print(f"分块后：{len(chunks)} 块")

    try:
        # 初始化向量数据库
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L12-v2"
        )
        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory=str(CHROMA_DIR),
        )
        # 向量检索
        vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        # BM25 检索
        bm25_retriever = BM25Retriever.from_documents(chunks)
        bm25_retriever.k = 3
        if vector_retriever and bm25_retriever:
            try:
                ensemble_retriever = EnsembleRetriever(
                    retrievers=[vector_retriever, bm25_retriever],
                    weights=[0.5, 0.5]
                )
                test_queries = [
                    ("语义查询", "LangChain 的功能"),
                    ("精确查询", "langchain>=1.0.0"),
                    ("混合查询", "BM25 算法原理"),
                ]
                for query_type, query in test_queries:
                    print(f"\n[INFO] {query_type}：{query}")
                    # BM25 结果
                    bm25_results = bm25_retriever.invoke(query)
                    bm25_previews = bm25_results[0].page_content[:40].replace("\n", " ") if bm25_results else "无"
                    # 向量 结果
                    vector_results = vector_retriever.invoke(query)
                    vector_previews = vector_results[0].page_content[:40].replace("\n", " ") if vector_results else "无"
                    # 混合 结果
                    ensemble_results = ensemble_retriever.invoke(query)
                    ensemble_previews = ensemble_results[0].page_content[:40].replace("\n", " ") if ensemble_results else "无"

                    print(f"BM25 结果：{bm25_previews}")
                    print(f"向量结果：{vector_previews}")
                    print(f"混合结果：{ensemble_previews}")
            except Exception as e:
                print(f"\n[ERROR] 混合检索失败：{e}")
    except Exception as e:
        print(f"\n[ERROR] 向量检索失败：{e}")

In [24]:
def main():
    print("\n"+"="*70)
    print("rag_advanced")
    print("="*70)

    example()

In [27]:
if __name__ == "__main__":
    main()


rag_advanced

[OK] 文档加载分块完成
原文档：1 个
分块后：4 块


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3865.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[INFO] 语义查询：LangChain 的功能
BM25 结果：LangChain 框架核心组件     1.Models - 模型接口    
向量结果：LangChain 框架核心组件     1.Models - 模型接口    
混合结果：LangChain 框架核心组件     1.Models - 模型接口    

[INFO] 精确查询：langchain>=1.0.0
BM25 结果：3.EnsembleRetriever     使用 RRF (Reciproc
向量结果：LangChain 框架核心组件     1.Models - 模型接口    
混合结果：LangChain 框架核心组件     1.Models - 模型接口    

[INFO] 混合查询：BM25 算法原理
BM25 结果：RAG 进阶技术     1.混合检索 (Hybrid Search)     
向量结果：RAG 进阶技术     1.混合检索 (Hybrid Search)     
混合结果：RAG 进阶技术     1.混合检索 (Hybrid Search)     
